# Qwen3.8-27B（现代多模态 Web 工作台 + OpenAI & Claude Code API）

- **支持显卡**：A100 (40GB / 80GB)、H100 (80GB)、RTX 6000 (96GB)
- **底层引擎**：原生 `vLLM 0.32.0`（80GB 显卡加载 [`Qwen/Qwen3.8-27B`](https://huggingface.co/Qwen/Qwen3.8-27B) BF16，40GB 显卡加载 FP8；`262144` 上下文 + MTP 投机解码）
- **单页全能能力**：多轮图像/视频多模态对话、实时可折叠 `<think>` 深度思考流、全采样超参调节、工具调用测试，同时原生兼容 **Claude Code (`/v1/messages`)** 与 **OpenAI SDK (`/v1/chat/completions`)**
- **访问方式**：默认使用 Cloudflare Quick Tunnel 临时公网域名（用完即关）；亦可在第 4 步切换「自有 Tunnel」从 Colab Secrets 读取 `CF_TUNNEL_TOKEN` 挂载固定域名
- **执行顺序**：`1. 环境检测` → `2. 安装 vLLM 推理引擎与应用服务包` → `3. 下载并校验 Qwen3.8-27B 模型权重` → `4. 启动现代多模态 Web 工作台与 API 服务`

| 入口路径 | 协议 / 说明 | 适用客户端 |
| --- | --- | --- |
| `GET /` | 现代亮色多模态对话与推理工作台（支持拖拽/粘贴图片与视频、实时思考链与机器资源监控） | 浏览器直接访问 |
| `POST /v1/messages` | Anthropic Messages 原生协议 | **Claude Code (`claude --model qwen3.8-27b`)** |
| `POST /v1/chat/completions` | OpenAI Chat Completions 协议 | OpenAI SDK、Cursor、Cline、Cherry Studio |
| `GET /v1/models` | 模型列表查询 | 模型名 `qwen3.8-27b` 或 `Qwen/Qwen3.8-27B` |
| `GET /docs` | Swagger / OpenAPI 在线交互接口文档 | 浏览器 / 开发者调试 |

### 1. 环境检测
- **功能**：检测 GPU 型号、显存容量、BF16 支持、系统内存、磁盘空间及 PyTorch / CUDA 版本。
- **要求**：支持 `A100`、`H100`、`RTX 6000`，显存 40GB 及以上。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 1. 环境检测 {display-mode: "form"}
import os, shutil, subprocess, torch

MIN_VRAM_GIB = 35
REQUIREMENT = "40GB 及以上显存（A100 40GB / 80GB、H100 80GB、RTX 6000 96GB）"

if not torch.cuda.is_available():
    raise RuntimeError("未检测到 GPU。请在 Colab 菜单选择：代码执行程序 -> 更改运行时类型 -> A100 / H100 / RTX 6000 GPU。")

print(subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout)

gpu = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
ram_gib = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
disk_gib = shutil.disk_usage("/content").free / 1024**3
bf16 = torch.cuda.is_bf16_supported()
print(f"GPU: {gpu} | 显存: {vram_gib:.1f} GiB | 系统内存: {ram_gib:.1f} GiB | 磁盘可用: {disk_gib:.1f} GiB | BF16: {bf16}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

if not any(k in gpu.upper() for k in ("A100", "H100", "6000")) or vram_gib < MIN_VRAM_GIB:
    raise RuntimeError(f"当前显卡为 {gpu} ({vram_gib:.1f} GiB)，本笔记本需要 {REQUIREMENT}，请切换 runtime。")
if not bf16:
    raise RuntimeError("当前 GPU 不支持 BF16，请切换 runtime。")
if vram_gib < 70:
    print("[提示] 40GB 级显卡，第 3 步将自动选择 Qwen3.8-27B-FP8 模型。")
print(f"[OK] GPU 环境正常：{gpu} ({vram_gib:.1f} GiB)")

### 2. 安装 vLLM 推理引擎与应用服务包
- **功能**：安装 vLLM `0.32.0`、FastAPI 服务依赖、`hf_xet`、`openai`、`cloudflared` `2026.10.6`，并拉取校验 GitHub Release 发布的 `app-qwen3.8-27b.tar.gz`（`v0.2.2`，内含多模态对话前端工作台与网关服务）。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 2. 安装 vLLM 推理引擎与应用服务包 {display-mode: "form"}
import hashlib, shutil, subprocess, sys, tarfile, tempfile, urllib.request
from pathlib import Path

VLLM_VERSION = "0.32.0"
CLOUDFLARED_VERSION = "2026.10.6"

APP_DIR = Path("/content/app")
RELEASE_REPO = "boltguo/colab"
RELEASE_TAG = "v0.2.2"
APP_ARCHIVE = "app-qwen3.8-27b.tar.gz"
APP_SHA256 = "277184211451f644ddbf2a339e9654efb6b7636bbedee9c30fb15d848ddd26b3"

APP_SERVER_PACKAGES = [
    f"vllm=={VLLM_VERSION}",
    "fastapi>=0.115.0,<1.0.0",
    "uvicorn[standard]>=0.30.0,<1.0.0",
    "python-multipart>=0.0.9,<1.0.0",
    "pydantic>=2.7.0,<3.0.0",
    "huggingface_hub>=0.23.0",
    "hf_xet",
    "openai",
]

def run(args, cwd=None):
    subprocess.run(args, cwd=cwd, check=True)

def ensure_not_symlink(path, label):
    for p in [path, *path.parents]:
        if p == Path("/content"):
            break
        if p.is_symlink():
            raise RuntimeError(f"{label} 路径包含软链接，请使用新的 runtime：{p}")

def install_release_app():
    ensure_not_symlink(APP_DIR, "应用目录")
    stamp_file = APP_DIR / ".release_sha256"
    if (
        stamp_file.is_file()
        and not stamp_file.is_symlink()
        and stamp_file.read_text(encoding="utf-8").strip() == f"{RELEASE_TAG}:{APP_SHA256}"
        and (APP_DIR / "backend/run_server.py").is_file()
    ):
        print(f"应用包已是最新：{APP_ARCHIVE} ({RELEASE_TAG})")
        return

    local_archive = Path(f"/content/{APP_ARCHIVE}")
    with tempfile.NamedTemporaryFile(suffix=".tar.gz", delete=False) as tmp:
        tmp_path = Path(tmp.name)
    try:
        if local_archive.is_file() and not local_archive.is_symlink():
            print(f"检测到本地预置应用包：{local_archive}")
            payload = local_archive.read_bytes()
        else:
            url = f"https://github.com/{RELEASE_REPO}/releases/download/{RELEASE_TAG}/{APP_ARCHIVE}"
            print(f"正在拉取应用包：{url}")
            req = urllib.request.Request(url, headers={"User-Agent": "colab-release-fetcher"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                payload = resp.read()
        actual_sha = hashlib.sha256(payload).hexdigest()
        if actual_sha != APP_SHA256:
            raise RuntimeError(
                f"应用包 SHA256 校验失败！预期 {APP_SHA256}，实际 {actual_sha}"
            )
        tmp_path.write_bytes(payload)

        staging = APP_DIR.with_name("app.staging")
        if staging.is_symlink():
            raise RuntimeError("staging 目录为软链接，拒绝写入。")
        if staging.exists():
            shutil.rmtree(staging)
        staging.mkdir(parents=True, exist_ok=True)
        staging_root = staging.resolve()

        with tarfile.open(tmp_path, "r:gz") as tf:
            for member in tf.getmembers():
                if member.issym() or member.islnk():
                    raise RuntimeError(f"拒绝解压含链接的归档条目：{member.name}")
                target = (staging / member.name).resolve()
                if target != staging_root and not str(target).startswith(str(staging_root) + "/"):
                    raise RuntimeError(f"归档路径越界：{member.name}")
            tf.extractall(staging)

        (staging / ".release_sha256").write_text(f"{RELEASE_TAG}:{APP_SHA256}\n", encoding="utf-8")
        if APP_DIR.exists():
            shutil.rmtree(APP_DIR)
        staging.replace(APP_DIR)
    finally:
        if tmp_path.exists():
            tmp_path.unlink()

run([sys.executable, "-m", "pip", "install", "-q", *APP_SERVER_PACKAGES])
install_release_app()

cloudflared_bin = Path("/usr/local/bin/cloudflared")
try:
    installed = subprocess.run([str(cloudflared_bin), "--version"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout
except OSError:
    installed = ""
if CLOUDFLARED_VERSION not in installed.split():
    download = cloudflared_bin.with_name("cloudflared.download")
    urllib.request.urlretrieve(
        f"https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64",
        str(download),
    )
    download.chmod(0o755)
    download.replace(cloudflared_bin)

vllm_ver = subprocess.check_output([sys.executable, "-c", "import vllm; print(vllm.__version__)"], text=True).strip()
print(f"Engine: vLLM {vllm_ver} | app: {RELEASE_TAG} ({APP_SHA256[:12]}) | cloudflared: {CLOUDFLARED_VERSION}")
print("[OK] vLLM 引擎与现代 Web/API 应用包安装完成。")

### 3. 下载 Qwen3.8-27B 模型
- **功能**：按显存下载模型至 `/content/models`：80GB 及以上为 `Qwen/Qwen3.8-27B` BF16（约 55 GB），40GB 为 `Qwen/Qwen3.8-27B-FP8`（约 28 GB）。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 3. 下载并校验 Qwen3.8-27B 模型权重 {display-mode: "form"}
import os, shutil, time, torch
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
os.environ.setdefault("HF_XET_CHUNK_CACHE_SIZE_BYTES", "0")
from huggingface_hub import snapshot_download

MODEL_SPECS = {
    "BF16": {
        "repo_id": "Qwen/Qwen3.8-27B",
        "revision": "1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0",
        "local_dir": Path("/content/models/Qwen3.8-27B"),
        "min_free_gb": 58,
    },
    "FP8": {
        "repo_id": "Qwen/Qwen3.8-27B-FP8",
        "revision": "017b9c7af6b5689d5dd426a76e0bc077eb5ca20a",
        "local_dir": Path("/content/models/Qwen3.8-27B-FP8"),
        "min_free_gb": 31,
    },
}

def ensure_not_symlink(path: Path) -> None:
    for p in [path, *path.parents]:
        if p == Path("/content"):
            break
        if p.is_symlink():
            raise RuntimeError(f"拒绝访问软链接模型路径：{p}")

vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
precision = "BF16" if vram_gib >= 70 else "FP8"
spec = MODEL_SPECS[precision]
target_dir = spec["local_dir"]
ensure_not_symlink(target_dir)
target_dir.parent.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage(target_dir.parent).free / 1024**3
if not (target_dir / "config.json").is_file() and free_gb < spec["min_free_gb"]:
    raise RuntimeError(f"磁盘空间不足：{spec['repo_id']} 需 {spec['min_free_gb']} GB，可用 {free_gb:.1f} GB。")

print(f"下载：{spec['repo_id']}（{precision}，显存 {vram_gib:.1f} GiB，commit {spec['revision'][:12]}）")
started = time.time()
snapshot_download(
    repo_id=spec["repo_id"],
    revision=spec["revision"],
    local_dir=str(target_dir),
    token=False,
)
if target_dir.is_symlink() or (target_dir / "config.json").is_symlink():
    raise RuntimeError("模型目录校验异常（检测到软链接）。")

SELECTED_QWEN38_DIR = str(target_dir)
marker = Path("/content/models/.selected_qwen38_dir")
if not marker.is_symlink():
    marker.write_text(SELECTED_QWEN38_DIR, encoding="utf-8")
print(f"[OK] 模型准备完成：{target_dir}（用时 {time.time() - started:.0f} 秒）")

### 4. 启动现代多模态 Web 工作台与 OpenAI / Claude Code API 服务
- **功能**：后台启动内部 `vllm serve`（`127.0.0.1:8189`，`262144` 上下文，MTP 投机采样）与对外 FastAPI 网关（`127.0.0.1:8188`，同时承载 `/` 多模态对话工作台、OpenAI `/v1/chat/completions`、Anthropic `/v1/messages` 与 `/docs`）。
- **隧道选择**：`TUNNEL_MODE` 默认选择 `Quick Tunnel (临时链接)`（免费公网临时域名，实例关闭即失效）；也可切换为 `自有 Tunnel`，从 Colab 左侧 **🔑 Secrets** 读取 `CF_TUNNEL_TOKEN` 挂载固定域名。
- **访问密码**：在下方 `ACCESS_PASSWORD` 填写应用访问密码，或在 Secrets 配置同名项。登录网页后，在右上角「API Key」中单独设置 OpenAI / Claude Code 接口密钥。共享笔记本时建议使用 Secrets。
- **使用**：直接运行，点击公网链接打开网页对话，或复制输出的环境变量直连 Claude Code / OpenAI SDK。

In [ ]:
# @title 4. 启动现代 Web 工作台与 OpenAI / Claude Code API 服务 {display-mode: "form"}
TUNNEL_MODE = "Quick Tunnel (临时链接)"  # @param ["Quick Tunnel (临时链接)", "自有 Tunnel (读取 Secrets 中 CF_TUNNEL_TOKEN)"]
ACCESS_PASSWORD = ""  # @param {type:"string"}
USE_CF_TUNNEL_TOKEN = TUNNEL_MODE.startswith("自有")
import hashlib, json, os, re, subprocess, sys, time, urllib.request
from pathlib import Path

FEATURE_NAME = "qwen3.8-27b"
VLLM_PORT = 8189
VLLM_BASE = f"http://127.0.0.1:{VLLM_PORT}"
VLLM_LOG = Path("/content/vllm-worker.log")
VLLM_LAUNCH_CONFIG = Path("/content/vllm-worker.launch.json")
VLLM_PROCESS_PATTERN = r"vllm serve|vllm\.entrypoints"

APP_DIR = Path("/content/app")
DATA_DIR = Path(f"/content/app_data/{FEATURE_NAME}")
UPLOADS_DIR = DATA_DIR / "uploads"
PORT = 8188
BASE = f"http://127.0.0.1:{PORT}"
APP_LOG = Path(f"/content/{FEATURE_NAME}-web.log")
APP_LAUNCH_CONFIG = Path(f"/content/{FEATURE_NAME}.launch.json")
APP_PROCESS_PATTERN = r"backend/run_server\.py|backend\.run_server"

CF_BIN = Path("/usr/local/bin/cloudflared")
CF_LOG = Path("/content/cloudflared.log")

def read_text(path):
    return path.read_text(errors="replace") if path.exists() else ""

def tail(path, size=8000):
    return read_text(path)[-size:]

def rotate_log(path):
    if path.is_file() and path.stat().st_size:
        path.replace(path.with_name(path.name + ".1"))

def process_running(pattern):
    return subprocess.run(["pgrep", "-f", pattern], stdout=subprocess.DEVNULL).returncode == 0

def stop_processes(pattern, timeout):
    subprocess.run(["pkill", "-f", pattern], check=False)
    deadline = time.time() + timeout
    while process_running(pattern) and time.time() < deadline:
        time.sleep(2)
    if process_running(pattern):
        subprocess.run(["pkill", "-9", "-f", pattern], check=False)
        time.sleep(2)

def resolve_cf_token():
    t = ""
    try:
        from google.colab import userdata
        t = (userdata.get("CF_TUNNEL_TOKEN") or "").strip()
    except Exception:
        t = ""
    if not t:
        t = os.environ.get("CF_TUNNEL_TOKEN", "").strip()
    return t.split()[-1] if t else ""

def start_cf_tunnel(token=""):
    if not CF_BIN.exists():
        raise RuntimeError("未找到 cloudflared，请先运行第 2 步。")
    stop_processes("cloudflared tunnel", 10)
    rotate_log(CF_LOG)
    cmd = [
        str(CF_BIN),
        "tunnel",
        "--no-autoupdate",
        "--protocol", "http2",
        "--edge-ip-version", "4",
    ]
    if token:
        cmd.extend(["run", "--url", BASE, "--token", token])
    else:
        cmd.extend(["--url", BASE])
    with CF_LOG.open("w") as cf_log:
        cf_proc = subprocess.Popen(
            cmd,
            stdin=subprocess.DEVNULL,
            stdout=cf_log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    deadline = time.time() + (30 if token else 60)
    while time.time() < deadline:
        if cf_proc.poll() is not None:
            print(tail(CF_LOG, 4000))
            if token:
                raise RuntimeError("Cloudflare Tunnel 启动失败，请检查 CF_TUNNEL_TOKEN 是否正确。")
            raise RuntimeError("Cloudflare 临时隧道启动失败，请查看上方日志。")
        log_text = read_text(CF_LOG)
        if not token:
            match = re.search(r"https://(?!api\.)[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
            if match and "Registered tunnel connection" in log_text:
                url = match.group(0)
                print(f"[OK] Cloudflare 临时隧道已连接 (HTTP/2): {url} -> {BASE}")
                return url
        elif "Registered tunnel connection" in log_text:
            if "Updated to new configuration" not in log_text:
                time.sleep(1)
                log_text = read_text(CF_LOG)
            configs = [line for line in log_text.splitlines() if "Updated to new configuration" in line]
            config = configs[-1].replace('\\"', '"') if configs else ""
            routes = re.findall(r'"hostname"\s*:\s*"([^"]+)"(?:(?!"hostname").)*?"service"\s*:\s*"([^"]+)"', config)
            for host, service in routes:
                print(f"[OK] Cloudflare Tunnel 已连接 (HTTP/2): https://{host} -> {service}")
            hosts = [host for host, service in routes if f":{PORT}" in service]
            if not routes:
                print("[OK] Cloudflare Tunnel 已连接 (HTTP/2)，请通过你在 Cloudflare 绑定的域名访问。")
            elif not hosts:
                print(f"[WARN] 没有域名转发到本服务的 {PORT} 端口！")
                print(f"       请前往 Cloudflare Zero Trust -> Networks -> Tunnels -> Public Hostname，将 Service URL 改为 {BASE}")
            return f"https://{hosts[0]}" if hosts else "https://<你的CF域名>"
        time.sleep(1)
    if not token:
        match = re.search(r"https://(?!api\.)[a-zA-Z0-9-]+\.trycloudflare\.com", read_text(CF_LOG))
        if match:
            url = match.group(0)
            print(f"[OK] Cloudflare 临时隧道已创建: {url} -> {BASE}")
            return url
        print(tail(CF_LOG, 4000))
        raise RuntimeError(f"Cloudflare 临时隧道 60 秒内未返回地址，详见 {CF_LOG}。")
    print(f"[WARN] Cloudflare Tunnel 30 秒内未确认连接，仍在后台运行，详见 {CF_LOG}。")
    return "https://<你的CF域名>"

def verify_web_password(base_url, process_pattern):
    from urllib.error import HTTPError

    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, req, fp, code, msg, headers, newurl):
            return None

    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}), NoRedirect())
    try:
        with opener.open(base_url + "/", timeout=10):
            pass
    except HTTPError as error:
        if error.code == 303 and error.headers.get("Location") == "/auth/login":
            print("[OK] 应用访问密码已生效。")
            return
    except Exception:
        pass
    stop_processes(process_pattern, 30)
    raise RuntimeError("应用访问密码未生效，已停止 Web 服务。请更新笔记本并重新运行第 2、4 步。")

def check_public_url(url, route):
    if "<" in url:
        return
    error = None
    deadline = time.time() + 60
    while time.time() < deadline:
        try:
            request = urllib.request.Request(url + route, headers={"User-Agent": "colab-tunnel-check"})
            with urllib.request.urlopen(request, timeout=10):
                print(f"[OK] 公网访问正常：{url}")
                return
        except Exception as e:
            error = e
        time.sleep(3)
    print(f"[WARN] 公网地址 60 秒内无法访问：{url}{route}（{error}）")

cf_token = resolve_cf_token() if USE_CF_TUNNEL_TOKEN else ""
if USE_CF_TUNNEL_TOKEN and not cf_token:
    raise RuntimeError("已勾选 USE_CF_TUNNEL_TOKEN，但未检测到 CF_TUNNEL_TOKEN；请在 Colab 左侧 🔑 Secrets 中添加 CF_TUNNEL_TOKEN 并开启访问权限，或取消勾选改用临时隧道。")

if APP_DIR.is_symlink() or not (APP_DIR / "backend/run_server.py").is_file():
    raise RuntimeError("未找到应用服务目录 /content/app，请先运行第 2 步。")

marker = Path("/content/models/.selected_qwen38_dir")
model_dir = (
    globals().get("SELECTED_QWEN38_DIR")
    or (marker.read_text(encoding="utf-8").strip() if marker.is_file() and not marker.is_symlink() else "")
    or ("/content/models/Qwen3.8-27B" if Path("/content/models/Qwen3.8-27B/config.json").is_file() else "/content/models/Qwen3.8-27B-FP8")
)
if Path(model_dir).is_symlink() or not (Path(model_dir) / "config.json").is_file():
    raise RuntimeError(f"未找到模型目录 {model_dir}，请先运行第 3 步下载模型。")

def resolve_access_password():
    if ACCESS_PASSWORD.strip():
        return ACCESS_PASSWORD.strip()
    key = ""
    try:
        from google.colab import userdata
        key = (userdata.get("ACCESS_PASSWORD") or "").strip()
    except Exception:
        key = ""
    if not key:
        key = os.environ.get("ACCESS_PASSWORD", "").strip()
    return key

access_password = resolve_access_password()
if not access_password:
    raise RuntimeError(
        "请在 ACCESS_PASSWORD 填写应用访问密码，或在 Colab 左侧 🔑 Secrets 配置同名项。"
    )

if DATA_DIR.is_symlink() or UPLOADS_DIR.is_symlink():
    raise RuntimeError("数据目录包含软链接，拒绝启动。")
UPLOADS_DIR.mkdir(parents=True, exist_ok=True)

# 1. 启动内部 vLLM 引擎 (127.0.0.1:8189，不对公网直接暴露)
def vllm_ready(timeout=5):
    try:
        with urllib.request.urlopen(VLLM_BASE + "/health", timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False

vllm_cmd = [
    "vllm", "serve", model_dir,
    "--served-model-name", "Qwen/Qwen3.8-27B", "qwen3.8-27b",
    "--host", "127.0.0.1",
    "--port", str(VLLM_PORT),
    "--max-model-len", "262144",
    "--gpu-memory-utilization", "0.93",
    "--enable-prefix-caching",
    "--reasoning-parser", "qwen3",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "qwen3_coder",
    "--allowed-local-media-path", str(UPLOADS_DIR),
    "--trust-remote-code",
    "--speculative-config", '{"method":"mtp","num_speculative_tokens":3}',
]

vllm_cfg_str = json.dumps(vllm_cmd)
vllm_is_up = vllm_ready() or (process_running(VLLM_PROCESS_PATTERN) and vllm_ready(timeout=30))
if vllm_is_up and read_text(VLLM_LAUNCH_CONFIG) != vllm_cfg_str:
    print("vLLM 启动参数已变化，正在按新参数重启...")
    stop_processes(VLLM_PROCESS_PATTERN, 120)
    vllm_is_up = False

if vllm_is_up:
    print("[OK] 内部 vLLM 推理引擎已在运行 (127.0.0.1:8189)。")
else:
    stop_processes(VLLM_PROCESS_PATTERN, 120)
    rotate_log(VLLM_LOG)
    with VLLM_LOG.open("w") as log:
        vllm_proc = subprocess.Popen(
            vllm_cmd,
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    VLLM_LAUNCH_CONFIG.write_text(vllm_cfg_str, encoding="utf-8")
    print("正在加载 27B 权重并初始化 256K KV Cache（约需 1~3 分钟）...")
    deadline = time.time() + 900
    while time.time() < deadline:
        if vllm_proc.poll() is not None:
            print(tail(VLLM_LOG))
            raise RuntimeError(f"vLLM 启动失败（退出码 {vllm_proc.returncode}），请查看上方日志。")
        if vllm_ready():
            print("[OK] 内部 vLLM 推理引擎启动成功 (127.0.0.1:8189)。")
            break
        time.sleep(3)
    else:
        print(tail(VLLM_LOG))
        raise TimeoutError("vLLM 15 分钟内未就绪，请查看上方日志。")

# 2. 启动对外 FastAPI + 现代多模态前端工作台 + OpenAI/Anthropic 网关 (127.0.0.1:8188)
def app_ready(timeout=5):
    try:
        with urllib.request.urlopen(BASE + "/api/v1/health", timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False

env_overrides = {
    "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1",
    "COLAB_FEATURE": FEATURE_NAME,
    "COLAB_HOST": "127.0.0.1",
    "COLAB_PORT": str(PORT),
    "COLAB_DATA_DIR": str(DATA_DIR),
    "COLAB_DIST_DIR": str(APP_DIR / "dist" / FEATURE_NAME),
    "COLAB_ACCESS_PASSWORD": access_password,
    "COLAB_API_KEY": "",
    "VLLM_BASE_URL": VLLM_BASE,
    "VLLM_ALLOW_LOCAL_MEDIA": "1",
}

app_cmd = [
    sys.executable,
    str(APP_DIR / "backend/run_server.py"),
    "--feature",
    FEATURE_NAME,
    "--host",
    "127.0.0.1",
    "--port",
    str(PORT),
    "--dist-dir",
    str(APP_DIR / "dist" / FEATURE_NAME),
]

app_fingerprint = json.dumps({
    "cmd": app_cmd,
    "release": read_text(APP_DIR / ".release_sha256").strip(),
    "web_auth_sha256": hashlib.sha256(access_password.encode("utf-8")).hexdigest(),
    "env": {key: value for key, value in env_overrides.items() if key not in {"COLAB_ACCESS_PASSWORD", "COLAB_API_KEY"}},
}, sort_keys=True)
app_is_up = app_ready() or (process_running(APP_PROCESS_PATTERN) and app_ready(timeout=15))
if app_is_up and read_text(APP_LAUNCH_CONFIG) != app_fingerprint:
    print("Web/API 网关配置已变更，正在重启...")
    stop_processes(APP_PROCESS_PATTERN, 30)
    app_is_up = False

if app_is_up:
    print("[OK] 现代 Web 工作台与 OpenAI/Anthropic API 网关已在运行 (127.0.0.1:8188)。")
else:
    stop_processes(APP_PROCESS_PATTERN, 30)
    rotate_log(APP_LOG)
    child_env = {**os.environ, **env_overrides}
    with APP_LOG.open("w") as log:
        app_proc = subprocess.Popen(
            app_cmd,
            cwd=APP_DIR,
            env=child_env,
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    APP_LAUNCH_CONFIG.write_text(app_fingerprint, encoding="utf-8")
    print("正在启动现代多模态 Web 工作台与 API 网关...")
    deadline = time.time() + 120
    while time.time() < deadline:
        if app_proc.poll() is not None:
            print(tail(APP_LOG))
            raise RuntimeError(f"Web/API 服务启动失败（退出码 {app_proc.returncode}），详见上方日志。")
        if app_ready():
            print("[OK] 现代 Web 工作台与 API 网关启动成功。")
            break
        time.sleep(1.5)
    else:
        print(tail(APP_LOG))
        raise TimeoutError("Web/API 服务 2 分钟内未就绪，请检查日志。")

# 3. 挂载 Cloudflare Tunnel 并同步隧道元数据
verify_web_password(BASE, APP_PROCESS_PATTERN)
host_url = start_cf_tunnel(cf_token)
tunnel_mode_key = "custom" if cf_token else "quick"
tunnel_info_path = DATA_DIR / "tunnel_info.json"
if not tunnel_info_path.is_symlink():
    tunnel_info_path.write_text(
        json.dumps({"public_url": host_url, "tunnel_mode": tunnel_mode_key}, ensure_ascii=False),
        encoding="utf-8",
    )
check_public_url(host_url, "/api/v1/health")

key_display = "YOUR_API_KEY"
tunnel_tip = (
    "⚠️ 提示：当前使用 Cloudflare Quick Tunnel 临时公网域名，实例关闭或重连隧道后地址即失效；如需固定域名请在上方选择「自有 Tunnel」。"
    if tunnel_mode_key == "quick"
    else "✅ 当前使用自有 Cloudflare Tunnel 固定域名。"
)
print(f"""
=== 访问入口 ===
🌐 现代多模态 Web 工作台 : {host_url}
📖 在线 API 文档         : {host_url}/docs
🩺 机器与显存实时状态    : {host_url}/api/v1/system
{tunnel_tip}
🔑 API Key：登录网页后，在右上角「API Key」中单独设置。

=== 1. Claude Code (cc) 直连配置（Anthropic /v1/messages 原生协议）===
export ANTHROPIC_BASE_URL="{host_url}"
export ANTHROPIC_AUTH_TOKEN="{key_display}"
export ANTHROPIC_API_KEY="{key_display}"
export ANTHROPIC_MODEL="qwen3.8-27b"
export ANTHROPIC_SMALL_FAST_MODEL="qwen3.8-27b"
export CLAUDE_CODE_DISABLE_NONESSENTIAL_TRAFFIC=1
claude --model qwen3.8-27b

=== 2. OpenAI 兼容接口配置（/v1/chat/completions）===
Base URL : {host_url}/v1
Model    : qwen3.8-27b （或 Qwen/Qwen3.8-27B）
API Key  : {key_display}
""")